# Banking77: how far can a decision model be pushed on option count?

Accuracy against the number of options k, for Laya (421M decision model) and
Qwen2.5-0.5B-Instruct (generative baseline), on identical examples.

Runs on **Kaggle or Colab** -- it picks its own paths.

* Kaggle: sidebar -> Accelerator **GPU T4 x2**, Internet **On**
* Colab: Runtime -> Change runtime type -> **T4 GPU**

Then run cell 1, **restart the session**, and run everything from cell 2 down.
(Kaggle: Run -> Restart session. Colab: Runtime -> Restart session.)

The restart is not optional. Installing `laya` replaces torch on disk while the
old torch is already imported, and every later import dies with
`module 'torch' has no attribute '_utils'`.

In [ ]:
!pip install -q laya
print("installed -- now Runtime -> Restart session, then continue from the next cell")

---
## 1. Setup

Everything below assumes the runtime was restarted after the install above.

In [ ]:
import os

# One notebook for both platforms.
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO = BASE + "/HARNESS"

%cd $BASE
!rm -rf HARNESS && git clone --depth 1 https://github.com/Vansh-kap-98/HARNESS.git HARNESS
%cd $REPO
print("working in", REPO)

In [ ]:
import json, os, random, sys, time

sys.path.insert(0, REPO)

import pandas as pd
import torch

from src.compiler import MAX_CHOICE_OPTIONS
from src.score import Proportion, calibration
from src.sweep import (Operation, RoutingExample, approx_tokens_by_chars,
                       build_question, question_tokens, score_k)

# ---- configuration; all of this goes into the saved run record --------------
SEED = 0
SWEEP_N = 500                       # examples per k
KS = (4, 8, 16, 32, 64, 77)
QWEN_KS = (4, 8, 16, 32)            # k=64 is ~30 min and k=77 ~45 min: add them
                                    # once you know whether normalisation matters
CONTEXT = 512
BANKING_INSTRUCTIONS = "Which customer-service intent does this message express?"
BANKING77_TEST = ("https://raw.githubusercontent.com/PolyAI-LDN/"
                  "task-specific-datasets/master/banking_data/test.csv")

df = pd.read_csv(BANKING77_TEST)
LABELS = sorted(df["category"].unique())

catalogue = [Operation(id=name, description=name.replace("_", " ")) for name in LABELS]
pool = [RoutingExample(id="b%04d" % i, state=row.text, gold=row.category)
        for i, row in enumerate(df.itertuples(index=False))]

# The CSV is grouped by category, so an unshuffled head would cover ~12 intents.
random.Random(SEED).shuffle(pool)
subset = pool[:SWEEP_N]

print(len(df), "queries,", len(LABELS), "labels;", len(subset), "examples per k")

## 2. Token budget

Does the question actually fit in the 512-token context? Uses Laya's own
tokenizer. It sums state + instructions + each option id + each description and
omits Laya's per-option `[MASK]` and separators, so it is a lower bound on real
usage.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("convaiinnovations/laya", subfolder="tokenizer")
count_tokens = lambda text: len(tok.encode(text, add_special_tokens=False))

token_budget = {}
print("  k      est     real   over %d" % CONTEXT)
for k in KS:
    built = [build_question(e, catalogue, k, SEED, instructions=BANKING_INSTRUCTIONS)
             for e in subset]
    est = sorted(question_tokens(b, approx_tokens_by_chars) for b in built)
    real = sorted(question_tokens(b, count_tokens) for b in built)
    over = sum(1 for c in real if c > CONTEXT)
    token_budget[k] = {"median_estimated": est[len(est) // 2],
                       "median_real": real[len(real) // 2],
                       "over_context": over, "n": len(real)}
    print("%3d %8d %8d   %d/%d"
          % (k, est[len(est) // 2], real[len(real) // 2], over, len(real)))

## 3. Laya

Watch the load-time warning about invalid temperatures: the checkpoint ships an
out-of-range temperature for the `choice:11+` bucket, so confidences at k>=11
are uncalibrated by the library's own admission.

In [ ]:
from laya import Router

router = Router(preload=True, device="cuda")

laya_results, laya_preds, laya_conf = {}, {}, {}
for k in KS:
    built = [build_question(e, catalogue, k, SEED, instructions=BANKING_INSTRUCTIONS)
             for e in subset]
    preds, confs, t0 = [], [], time.time()
    for item in built:
        answer = router.predict(item.state, {"intent": item.question},
                                model="english")["answers"]["intent"]
        preds.append(answer["choice"])
        probs = answer["probabilities"]
        confs.append(max(probs.values()) if isinstance(probs, dict) else max(probs))
    laya_results[k] = score_k(built, preds, compiler_max_options=MAX_CHOICE_OPTIONS,
                              count_tokens=count_tokens, context_limit=CONTEXT)
    laya_preds[k], laya_conf[k] = preds, confs
    print("laya k=%3d  %s  chance %.4f  %.0fs"
          % (k, laya_results[k].accuracy, laya_results[k].chance, time.time() - t0))

In [ ]:
# Is the cliff an artefact of which distractors were sampled?
cliff = {}
for seed in (0, 1):
    for k in (16, 32):
        built = [build_question(e, catalogue, k, seed, instructions=BANKING_INSTRUCTIONS)
                 for e in subset]
        preds = [router.predict(item.state, {"intent": item.question},
                                model="english")["answers"]["intent"]["choice"]
                 for item in built]
        cliff[(seed, k)] = score_k(built, preds, compiler_max_options=MAX_CHOICE_OPTIONS)
        print("seed=%d k=%2d  %s" % (seed, k, cliff[(seed, k)].accuracy))

for seed in (0, 1):
    print("seed=%d  16->32 drop: %.3f"
          % (seed, cliff[(seed, 16)].accuracy.value - cliff[(seed, 32)].accuracy.value))

In [ ]:
# Calibration at the largest k. Always quote it with the invalid-temperature caveat.
k = max(KS)
built = [build_question(e, catalogue, k, SEED, instructions=BANKING_INSTRUCTIONS)
         for e in subset]
correct = [p == item.gold for p, item in zip(laya_preds[k], built)]
cal = calibration(laya_conf[k], correct)
print("laya k=%d  ECE %.3f  (reportable: %s, n=%d)"
      % (k, cal.ece, cal.reportable, cal.total))

## 4. Qwen2.5-0.5B baseline

Every option is scored as a continuation of a prompt listing all k options, and
the highest-scoring option wins. For a closed set this is *stronger* than
grammar-constrained greedy decoding, which picks token by token and can walk
into a suboptimal option, so the baseline gets the better version of the task.

Two rankings come out of the same forward passes:

* **summed** log-probability, the faithful analogue of constrained decoding, but
  it favours short option names
* **mean** log-probability per token, length-normalised

If they disagree, the summed scorer was handicapping the baseline and the
normalised line is the one to report.

In [ ]:
from transformers import AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
CHUNK = 8                      # options per forward pass; lower this if it OOMs

qtok = AutoTokenizer.from_pretrained(MODEL_ID)
qmodel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda").eval()
print(MODEL_ID, "%.0fM params" % (sum(p.numel() for p in qmodel.parameters()) / 1e6))

In [ ]:
def build_prompt(state, question):
    listing = "\n".join("- %s: %s" % (name, text)
                        for name, text in question["criteria"].items())
    user = ("%s\n\nOptions:\n%s\n\nMessage: %s\n\n"
            "Answer with exactly one option name from the list."
            % (question["instructions"], listing, state))
    return qtok.apply_chat_template([{"role": "user", "content": user}],
                                    tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def option_scores(prompt, options):
    """(summed logprob, token count) for every option, in chunked passes."""
    prompt_ids = qtok(prompt, return_tensors="pt").input_ids[0]
    start = len(prompt_ids)
    pad = qtok.pad_token_id if qtok.pad_token_id is not None else qtok.eos_token_id
    totals, lengths = [], []
    for begin in range(0, len(options), CHUNK):
        group = options[begin:begin + CHUNK]
        seqs = [torch.cat([prompt_ids,
                           torch.tensor(qtok(o, add_special_tokens=False).input_ids)])
                for o in group]
        width = max(len(s) for s in seqs)
        batch = torch.full((len(seqs), width), pad, dtype=torch.long)
        mask = torch.zeros((len(seqs), width), dtype=torch.long)
        for i, seq in enumerate(seqs):
            batch[i, :len(seq)] = seq
            mask[i, :len(seq)] = 1
        batch, mask = batch.to(qmodel.device), mask.to(qmodel.device)
        logits = qmodel(input_ids=batch, attention_mask=mask).logits[:, start - 1:, :]
        logprobs = torch.log_softmax(logits.float(), dim=-1)
        for i, seq in enumerate(seqs):
            picked = [logprobs[i, pos - start, batch[i, pos]]
                      for pos in range(start, len(seq))]
            totals.append(float(torch.stack(picked).sum()))
            lengths.append(len(seq) - start)
    return totals, lengths

In [ ]:
qwen_sum, qwen_mean, shortest_rate = {}, {}, {}

for k in QWEN_KS:
    built = [build_question(e, catalogue, k, SEED, instructions=BANKING_INSTRUCTIONS)
             for e in subset]
    preds_sum, preds_mean, shortest, t0 = [], [], 0, time.time()
    for item in built:
        options = list(item.values)
        totals, lengths = option_scores(build_prompt(item.state, item.question), options)
        by_sum = max(range(len(options)), key=lambda i: totals[i])
        by_mean = max(range(len(options)), key=lambda i: totals[i] / lengths[i])
        preds_sum.append(options[by_sum])
        preds_mean.append(options[by_mean])
        shortest += lengths[by_sum] == min(lengths)
    qwen_sum[k] = score_k(built, preds_sum, compiler_max_options=MAX_CHOICE_OPTIONS)
    qwen_mean[k] = score_k(built, preds_mean, compiler_max_options=MAX_CHOICE_OPTIONS)
    shortest_rate[k] = shortest / len(built)
    print("qwen k=%3d  summed %s  normalised %s  picked-shortest %.1f%% (chance %.1f%%)  %.0fs"
          % (k, qwen_sum[k].accuracy, qwen_mean[k].accuracy,
             100 * shortest_rate[k], 100.0 / k, time.time() - t0))

## 5. Chart and record

If `picked-shortest` sits near `chance`, length bias is not a problem and the
summed line is the one to report. If it is well above, report the normalised
line and say why.

In [ ]:
import matplotlib.pyplot as plt

RESULTS = os.path.join(REPO, "results")
os.makedirs(RESULTS, exist_ok=True)


def band(res, ks):
    value = [res[k].accuracy.value for k in ks]
    low = [value[i] - res[k].accuracy.low for i, k in enumerate(ks)]
    high = [res[k].accuracy.high - value[i] for i, k in enumerate(ks)]
    return value, [low, high]


plt.figure(figsize=(7.5, 4.8))
value, err = band(laya_results, KS)
plt.errorbar(list(KS), value, yerr=err, marker="o", capsize=3,
             label="laya 421M (decision)")
for res, label in ((qwen_sum, "qwen 0.5B (summed)"),
                   (qwen_mean, "qwen 0.5B (length-normalised)")):
    value, err = band(res, QWEN_KS)
    plt.errorbar(list(QWEN_KS), value, yerr=err, marker="s", capsize=3, label=label)
plt.plot(list(KS), [1.0 / k for k in KS], "--", color="gray", label="chance (1/k)")
plt.xscale("log", base=2)
plt.xticks(list(KS), [str(k) for k in KS])
plt.xlabel("number of options (k)")
plt.ylabel("accuracy")
plt.title("Banking77: accuracy vs option count (n=%d per k)" % len(subset))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "banking77_two_lines.png"), dpi=150)
plt.show()

In [ ]:
record = {
    "dataset": "banking77 test.csv (PolyAI-LDN/task-specific-datasets)",
    "n_per_k": len(subset),
    "seed": SEED,
    "instructions": BANKING_INSTRUCTIONS,
    "criteria_text": "label name, underscores replaced by spaces",
    "laya": {"checkpoint": "english", "zero_shot": True,
             "results": {str(k): laya_results[k].as_dict() for k in KS}},
    "laya_cliff_replication": {"seed%d_k%d" % (s, k): cliff[(s, k)].accuracy.as_dict()
                               for (s, k) in cliff},
    "laya_ece_at_max_k": {"k": max(KS), "ece": cal.ece, "n": cal.total,
                          "caveat": "choice:11+ ships an out-of-range temperature; "
                                    "the library clamps it and calls these "
                                    "confidences uncalibrated"},
    "qwen": {"model": MODEL_ID,
             "scoring": "argmax over complete options; summed and length-normalised",
             "summed": {str(k): qwen_sum[k].as_dict() for k in QWEN_KS},
             "normalised": {str(k): qwen_mean[k].as_dict() for k in QWEN_KS},
             "picked_shortest_rate": {str(k): shortest_rate[k] for k in QWEN_KS}},
    "token_budget_real_tokenizer": {str(k): token_budget[k] for k in KS},
}

path = os.path.join(RESULTS, "banking77_run.json")
chart = os.path.join(RESULTS, "banking77_two_lines.png")
open(path, "w").write(json.dumps(record, indent=2))
print(json.dumps(record, indent=2)[:2000])

try:
    from google.colab import files
    files.download(path)
    files.download(chart)
except ImportError:
    print("\nOn Kaggle: both files are under", RESULTS)
    print("Save Version, or grab them from the Output panel, before the session ends.")